## Contenido del word de que cambios hicimos


1. Corrección de Lógica: Atributos Media y Similitud
Problema: Un Artista o Playlist no "tienen" atributos propios, sino que son el promedio de sus canciones. El código actual no calcula esto, por lo que no puede comparar si un artista es "similar" al usuario.

Qué reemplazar: Sustituimos las clases Artista y Playlist por estas para calcular las medias:


In [ ]:
# Reemplaza las clases Artista y Playlist originales
class Artista:
    def __init__(self, nombre, fecha_nacimiento, canciones):
        self.nombre = nombre
        self.fecha_nacimiento = fecha_nacimiento
        self.canciones = canciones

    @property
    def atributos_medios(self):
        # Usamos programación funcional para promediar los diccionarios de atributos
        # Esto cumple con el requisito de "Uso de funciones de orden superior"
        if not self.canciones: return {}
        todas_caract = [c.atributos_sonoros for c in self.canciones] # O sentimentales, según se pida
        keys = todas_caract[0].keys()
        return {k: sum(c[k] for c in todas_caract) / len(todas_caract) for k in keys}

class Playlist:
    def __init__(self, titulo, fecha_creacion, canciones):
        self.titulo = titulo
        self.fecha_creacion = fecha_creacion
        self.canciones = canciones

    @property
    def atributos_medios(self):
        if not self.canciones: return {}
        todas_caract = [c.atributos_sonoros for c in self.canciones]
        keys = todas_caract[0].keys()
        return {k: sum(c[k] for c in todas_caract) / len(todas_caract) for k in keys}

En el código viejo: Un Artista tiene un nombre y una lista de canciones, pero no tiene forma de saber qué "estilo" tiene. Si el sistema quiere recomendar un artista similar a una canción de Jazz, el código viejo no puede calcular si el artista hace Jazz o Rock porque no tiene acceso a sus promedios.

En el código nuevo: El método atributos_medios permite que el artista se presente al recomendador diciendo: "Mira, mi ritmo medio es 120 y mi energía media es 0.8". Esto permite comparar al artista con el usuario.

Uso de @property (Encapsulación)
En el código nuevo usamos el decorador @property.

¿Qué hace? Permite que llames a artista.atributos_medios como si fuera un atributo normal (sin paréntesis), pero por detrás se ejecuta un cálculo en tiempo real.

¿Por qué es mejor? Si el artista añade una canción nueva a su lista, la "media" se actualizará automáticamente la próxima vez que la pidas. En el código viejo, si intentaras guardar la media en el __init__, se quedaría desactualizada para siempre.

2. Corrección del Patrón Strategy: Criterio Temporal
Problema: Si intentas acceder a fecha_creacion en un objeto Artista, el programa lanzará un AttributeError porque los artistas tienen fecha_nacimiento.

Cambiamos en R4 la clase class BusquedaTemporal por esta

In [ ]:
class BusquedaTemporal(IEstrategiaBusqueda):
    def buscar(self, items):
        if not items: return None
        # Corrección: Detectamos el atributo correcto según el tipo de objeto
        # Usamos una función lambda y sorted (estilo funcional)
        return sorted(
            items, 
            key=lambda x: getattr(x, 'fecha_creacion', getattr(x, 'fecha_nacimiento', None)),
            reverse=True
        )[0]

El error antes:
En nuestra estrategia de búsqueda teniamos:
return sorted(items,key=lambda x:x.fecha_creacion,reverse=True)[0]

Por qué falla: * Si items son canciones, funciona (tienen fecha_creacion).

Si items son playlists, funciona (tienen fecha_creacion).

PERO, si el sistema intenta recomendar un Artista, el programa buscará artista.fecha_creacion, no la encontrará (porque los artistas tienen fecha_nacimiento) y lanzará un error: AttributeError: 'Artista' object has no attribute 'fecha_creacion'.

Nuestro cambio, le dice a Python: "Busca 'fecha_creacion'. ¿No está? Pues busca 'fecha_nacimiento'. ¿Tampoco? Pues pon None".

3. Corrección de los Decoradores: Similitud Real
Problema: Los decoradores actuales eligen cualquier artista/playlist al azar o por fecha, pero no comprueban si se parecen a lo que el usuario escucha.
No queremos que el usuario reciba un artista al azar (que es lo que hacía el código de tu compañero). Queremos que el sistema filtre el catálogo para
encontrar quién se parece más a lo que el usuario está oyendo en su sesión actual.
Qué reemplazamos: Modificamos los métodos recomendar de DecoratorArtista y DecoratorPlaylist:

1. Corrección para DecoratorArtista
Este código utiliza la función de orden superior filter (exigencia del Tema 4) para comparar la media de la sesión del usuario con la media de los artistas.

In [ ]:
class DecoratorArtista(DecoratorRecomendacion):
    def recomendar(self):
        # 1. Obtenemos el diccionario base de la canción
        resultado = super().recomendar()
        
        # 2. Cálculo funcional de la media de la sesión (Requisito Funcional)
        media_sesion = 0.0
        if self.sesion:
            # Aplanamos todos los valores de atributos sonoros de todas las canciones
            valores = [val for c in self.sesion for val in c.atributos_sonoros.values()]
            if valores:
                media_sesion = sum(valores) / len(valores)

        # 3. Filtrado funcional (Requisito Funcional)
        # Filtramos artistas cuya media de sus canciones no diste más de 0.5 de la sesión
        artistas_similares = list(filter(
            lambda a: abs((sum(a.atributos_medios.values()) / len(a.atributos_medios)) - media_sesion) < 0.5 
            if a.atributos_medios else False,
            self.catalogo.artistas
        ))

        # 4. Aplicamos Strategy sobre el filtro
        if artistas_similares:
            artista = self.estrategia.buscar(artistas_similares)
            resultado["artista"] = artista.nombre
        else:
            resultado["artista"] = "No se encontraron artistas similares"
            
        return resultado

2. Corrección para DecoratorPlaylist
Hacemos lo mismo para las listas de reproducción. Se puede ver que el código es casi idéntico; esto es gracias al polimorfismo (ambas clases se comportan igual ante la propiedad atributos_medios).

In [ ]:
class DecoratorPlaylist(DecoratorRecomendacion):
    def recomendar(self):
        resultado = super().recomendar()
        
        # Calculamos media de sesión igual que arriba
        media_sesion = 0.0
        if self.sesion:
            valores = [val for c in self.sesion for val in c.atributos_sonoros.values()]
            if valores:
                media_sesion = sum(valores) / len(valores)

        # Filtrado de Playlists
        playlists_similares = list(filter(
            lambda p: abs((sum(p.atributos_medios.values()) / len(p.atributos_medios)) - media_sesion) < 0.5 
            if p.atributos_medios else False,
            self.catalogo.playlists
        ))

        if playlists_similares:
            playlist = self.estrategia.buscar(playlists_similares)
            resultado["playlist"] = playlist.titulo
        else:
            resultado["playlist"] = "No hay playlists similares"
            
        return resultado

4. Implementación de Asincronía (Simulación de Tiempo Real)

Problema: El sistema original procesaba las escuchas de forma instantánea y bloqueante, lo cual no es realista para un servicio de streaming y no cumplía con el requisito de "actualización en tiempo real".

Solución: Hemos migrado el flujo principal a un modelo asíncrono usando la librería asyncio.

Beneficio: Ahora el programa simula la duración de las canciones mediante await asyncio.sleep(). Esto permite que el sistema "espere" sin congelar el hilo de ejecución, permitiendo que en un futuro el servidor atienda otras peticiones mientras el usuario escucha música.

Resultado: Se cumple con el Requisito R2 y se eleva la calidad técnica del proyecto al usar un paradigma de programación concurrente moderno.

Este es el bloque de codigo que hemos puesto ahora donde estaba antes el codigo de prueba:

In [ ]:
# 1. Creamos una función asíncrona para simular que el usuario escucha música
async def simular_escucha_real(recomendador):
    # Lista de IDs de canciones que el usuario va a oír
    canciones_a_escuchar = [1, 2, 3] 
    
    for c_id in canciones_a_escuchar:
        print(f"-> Reproduciendo canción ID: {c_id}...")
        # Registramos la escucha en el sistema
        recomendador.registrar_escucha(c_id)
        
        # 'await' pausa esta función 1 segundo (simula el tiempo real) 
        # sin detener el resto del programa
        await asyncio.sleep(1) 
    
    print("\n[INFO] Sesión de escucha completada.")

# 2. Función principal asíncrona que coordina todo el sistema
async def main():
    try:
        # Inicialización
        catalogo = generar_catalogo(20) # Aumentamos un poco para tener más variedad
        recomendador = Recomendador.obtenerRecomendador(catalogo)
        
        # R2: Concurrencia (Simulación de escucha)
        await simular_escucha_real(recomendador)
        
        # R3: Chain of Responsibility (Estadísticos)
        print("\n[PROCESANDO ESTADÍSTICOS DE SESIÓN]")
        cadena = EstadisticoSonoro(sucesor=EstadisticoSentimental())
        cadena.manejar(recomendador.sesion)
        
        print("\n--- Generando recomendación inteligente ---")
        
        # R4: Strategy
        estrategia = BusquedaTemporal() # Probamos una diferente a la aleatoria
        
        # R3: Decorator (Construcción por capas)
        # 1. Recomendación básica (Canción)
        rec_objeto = RecomendacionBase(catalogo, recomendador.sesion, estrategia)
        
        # 2. Añadimos capa de Artista
        rec_objeto = DecoratorArtista(rec_objeto)
        
        # 3. Añadimos capa de Playlist
        rec_objeto = DecoratorPlaylist(rec_objeto)
        
        # Ejecución final
        final = rec_objeto.recomendar()
        print(f"\nResultado final para el usuario:")
        for clave, valor in final.items():
            print(f" > {clave.capitalize()}: {valor}")
        
    except Exception as e:
        print(f"\n[ERROR EN EL SISTEMA]: {e}")

# 3. Punto de entrada que arranca el bucle de eventos (Event Loop)
if __name__ == "__main__":
    asyncio.run(main())